In [1]:
from torch.utils.data import DataLoader, random_split

import os
import random
import torch
import warnings

import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

warnings.filterwarnings('ignore')

In [2]:
BATCH_SIZE = 256
LR = 0.001
ES_THRES = 5
SEED = 1234
OUTPUT_DIR = "./state_dicts/MNIST"

# Set Seed

In [3]:
def seed_everything(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    
    if torch.cuda.is_available(): 
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True

seed_everything(SEED)

# Reference

- https://dacon.io/competitions/official/235614/codeshare/1300
- https://github.com/bentrevett/pytorch-image-classification/blob/master/2_lenet.ipynb

# Download Datasets

In [4]:
from torchvision.datasets import MNIST
import torchvision.transforms as transforms

In [5]:
download_root = '../datasets/MNIST_DATASET'
train_data = MNIST(download_root, train=True, download=True)
mean = train_data.data.float().mean() / 255
std = train_data.data.float().std() / 255

print(f'Calculated mean: {mean}')
print(f'Calculated std: {std}')

Calculated mean: 0.13066047430038452
Calculated std: 0.30810779333114624


In [6]:
train_transforms = transforms.Compose([
                            transforms.RandomRotation(5, fill=(0,)),
                            transforms.RandomCrop(28, padding = 2),
                            transforms.Grayscale(num_output_channels=3),
                            transforms.Resize(32),
                            transforms.ToTensor(),
                            transforms.Normalize(mean = [mean], std = [std])])

test_transforms = transforms.Compose([
                            transforms.Grayscale(num_output_channels=3),
                            transforms.Resize(32),
                            transforms.ToTensor(),
                            transforms.Normalize(mean = [mean], std = [std])])

In [7]:
train_valid_dataset = MNIST(download_root, transform=train_transforms, train=True, download=True)
train_dataset, valid_dataset = random_split(train_valid_dataset, [54000, 6000])
test_dataset = MNIST(download_root, transform=test_transforms, train=False, download=True)

In [8]:
len(train_dataset)

54000

In [9]:
len(valid_dataset)

6000

# Utils

In [10]:
class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

# Models 

In [11]:
class FeatureExtractor(nn.Module):
    def __init__(self, in_channel_dim=3, out_channel_dim=16):
        super(FeatureExtractor, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channel_dim, out_channels=6, kernel_size=5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=out_channel_dim, kernel_size=5)
        
    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)
        
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)
        
#         x = x.view(-1, 16 * 5 * 5)

        return x

In [12]:
class Classifier(nn.Module):
    def __init__(self, in_dim, out_dim=10):
        super(Classifier, self).__init__()
        self.fc1 = nn.Linear(in_dim, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, out_dim)
        
    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        
        x = self.fc2(x)
        x = F.relu(x)
        
        x = self.fc3(x)
        
        return x

In [13]:
class LeNet(nn.Module):
    def __init__(self, in_channel_dim=1, out_dim=10):
        super(LeNet, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channel_dim, out_channels=6, kernel_size=5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)
        
        self.fc1 = nn.Linear(256, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, out_dim)
        
    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)
        
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)
        
        x = torch.flatten(x, 1)
        
        x = self.fc1(x)
        x = F.relu(x)
        
        x = self.fc2(x)
        x = F.relu(x)
        
        x = self.fc3(x)
        
        return x

# Train/Eval Functions

In [14]:
def calculate_accuracy(y_pred, y):
    top_pred = y_pred.argmax(1, keepdim = True)
    correct = top_pred.eq(y.view_as(top_pred)).sum()
    acc = correct.float() / y.shape[0]
    return acc

In [15]:
def train_loop_classifier(feature_extractor, classifier, train_loader, loss_func, optimizer, 
                          summary_loss, summary_acc=None, device=None):
    feature_extractor.train()
    classifier.train()
    
    for batch_idx, (data, target) in enumerate(train_loader):
        if device is not None:
            data, target = data.to(device), target.to(device)
            
        optimizer.zero_grad()
        feature = feature_extractor(data)
        feature = torch.flatten(feature, 1)
        output = classifier(feature)
        loss = loss_func(output, target)
        loss.backward()
        optimizer.step()
        
        summary_loss.update(loss.detach().item(), BATCH_SIZE)
        if summary_acc is not None:
            acc = calculate_accuracy(output, target)
            summary_acc.update(acc, BATCH_SIZE)
        
    return summary_loss, summary_acc

def eval_loop_classifier(feature_extractor, classifier, valid_loader, loss_func, optimizer, 
                         summary_loss, summary_acc=None, device=None):
    feature_extractor.eval()
    classifier.eval()
    
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(valid_loader):
            if device is not None:
                data, target = data.to(device), target.to(device)

            feature = feature_extractor(data)
            feature = torch.flatten(feature, 1)
            output = classifier(feature)

            loss = loss_func(output, target)

            summary_loss.update(loss.detach().item(), BATCH_SIZE)
            if summary_acc is not None:
                acc = calculate_accuracy(output, target)
                summary_acc.update(acc, BATCH_SIZE)

    return summary_loss, summary_acc

# Training with Early Stopping

In [16]:
def run_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    feature_extractor = FeatureExtractor()
    classifier = Classifier(400)

    feature_extractor.to(device)
    classifier.to(device)
    
    feature_extractor.train()
    classifier.train()

    criterion = nn.CrossEntropyLoss()
    criterion.to(device)
    
    model_parameters = list(feature_extractor.parameters()) + list(classifier.parameters())
    optimizer = optim.Adam(model_parameters, lr=LR)

    train_loader = DataLoader(dataset=train_dataset, 
                             batch_size=BATCH_SIZE,
                             shuffle=True)

    valid_loader = DataLoader(dataset=valid_dataset, 
                             batch_size=BATCH_SIZE,
                             shuffle=True)

    test_loader = DataLoader(dataset=test_dataset, 
                             batch_size=BATCH_SIZE,
                             shuffle=True)

    best_epoch = None
    best_loss = None
    epoch = 0
    es_count = 0
    while(True):
        epoch += 1
        summary_loss_train = AverageMeter()
        summary_acc_train = AverageMeter()
        summary_loss_valid = AverageMeter()
        summary_acc_valid = AverageMeter()

        summary_loss_train, summary_acc_train = train_loop_classifier(feature_extractor, classifier, train_loader, 
                                                   criterion, optimizer, summary_loss_train, summary_acc_train, device=device)
        summary_loss_valid, summary_acc_valid = eval_loop_classifier(feature_extractor, classifier, valid_loader, 
                                                   criterion, optimizer, summary_loss_valid, summary_acc_valid, device=device)

        print(f"[epoch]{epoch} [train loss]{summary_loss_train.avg} [train acc]{summary_acc_train.avg}  [valid loss]{summary_loss_valid.avg} [valid acc]{summary_acc_valid.avg} ")

        if best_loss is None:
            best_loss = summary_loss_valid.avg
            best_epoch = epoch

        if best_loss > summary_loss_valid.avg:
            best_loss = summary_loss_valid.avg
            best_epoch = epoch
            es_count = 0
        else:
            es_count += 1

        if es_count == ES_THRES:
            break

    print(f"Best epoch: {best_epoch}")
    return best_epoch

In [17]:
best_epoch = run_training()

[epoch]1 [train loss]0.5960568876063089 [train acc]0.8144463300704956  [valid loss]0.21971508612235388 [valid acc]0.9330357313156128 
[epoch]2 [train loss]0.16776116217058418 [train acc]0.9484683275222778  [valid loss]0.13526649555812278 [valid acc]0.9590076804161072 
[epoch]3 [train loss]0.12033180103270928 [train acc]0.9625357985496521  [valid loss]0.11391045370449622 [valid acc]0.9629139304161072 
[epoch]4 [train loss]0.09460849844590183 [train acc]0.9706445336341858  [valid loss]0.09416931277761857 [valid acc]0.9697498679161072 
[epoch]5 [train loss]0.07989598610283921 [train acc]0.9755393862724304  [valid loss]0.08161509898491204 [valid acc]0.9734002947807312 
[epoch]6 [train loss]0.06976398705588698 [train acc]0.9783743619918823  [valid loss]0.0766377590286235 [valid acc]0.9752371311187744 
[epoch]7 [train loss]0.06288957977181928 [train acc]0.9805378913879395  [valid loss]0.06043590314220637 [valid acc]0.9813058376312256 
[epoch]8 [train loss]0.06105384892691368 [train acc]0.981

# Second Training

In [18]:
def run_second_training(best_epoch):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    feature_extractor = FeatureExtractor()
    classifier = Classifier(400)

    feature_extractor.to(device)
    classifier.to(device)
    
    feature_extractor.train()
    classifier.train()
    
    train_valid_loader = DataLoader(dataset=train_valid_dataset, 
                             batch_size=BATCH_SIZE,
                             shuffle=True)

    criterion = nn.CrossEntropyLoss()
    model_parameters = list(feature_extractor.parameters()) + list(classifier.parameters())
    optimizer = optim.Adam(model_parameters, lr=LR)

    epoch = 0
    for _ in range(best_epoch):
        epoch += 1
        summary_loss_train = AverageMeter()
    #     summary_acc_train = AverageMeter()

        summary_loss_train, _ = train_loop_classifier(feature_extractor, classifier, train_valid_loader, 
                                                   criterion, optimizer, summary_loss_train, None, device=device)

        print(f"[epoch]{epoch} [train loss]{summary_loss_train.avg}")
    return feature_extractor, classifier

In [19]:
feature_extractor, classifier = run_second_training(best_epoch)

[epoch]1 [train loss]0.5595671398842589
[epoch]2 [train loss]0.14663327561413989
[epoch]3 [train loss]0.10698945709365479
[epoch]4 [train loss]0.08747047758165827
[epoch]5 [train loss]0.07838027971063523
[epoch]6 [train loss]0.07012202609726723
[epoch]7 [train loss]0.061356606857573735
[epoch]8 [train loss]0.05807521381673027
[epoch]9 [train loss]0.05489259688381819
[epoch]10 [train loss]0.050573474600752615
[epoch]11 [train loss]0.04788324355762055
[epoch]12 [train loss]0.045305650613884973
[epoch]13 [train loss]0.046850391738909355
[epoch]14 [train loss]0.04164868884501939
[epoch]15 [train loss]0.03853895665800318
[epoch]16 [train loss]0.039524260016673425
[epoch]17 [train loss]0.039368029497563836
[epoch]18 [train loss]0.03649270090254697
[epoch]19 [train loss]0.03445167735694571
[epoch]20 [train loss]0.032555069433564836
[epoch]21 [train loss]0.033991707584008254
[epoch]22 [train loss]0.03057205669898936


In [20]:
def run_test(feature_extractor, classifier):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    feature_extractor.eval()
    classifier.eval()
    
    test_loader = DataLoader(dataset=test_dataset, 
                             batch_size=BATCH_SIZE,
                             shuffle=True)
    
    criterion = nn.CrossEntropyLoss()
    
    summary_loss_test = AverageMeter()
    summary_acc_test = AverageMeter()

    summary_loss_test, summary_acc_test = eval_loop_classifier(feature_extractor, classifier, test_loader, 
                                               criterion, None, summary_loss_test, summary_acc_test, device=device)

    print(f"[test loss]{summary_loss_test.avg} [test acc]{summary_acc_test.avg}")

In [21]:
run_test(feature_extractor, classifier)

[test loss]0.03335214597173035 [test acc]0.98828125


In [22]:
torch.save(feature_extractor.state_dict(), os.path.join(OUTPUT_DIR, f"eature_extractor.pth"))
torch.save(classifier.state_dict(), os.path.join(OUTPUT_DIR, "classifier.pth"))